# 04 CPSC Plan-Level Analysis

Purpose: use the optional CPSC files for plan/contract-level opportunity analysis. This is deeper than MA SCP because it includes contract and plan identifiers.

Implementation decision: read the large CPSC enrollment CSVs in chunks. That avoids loading multi-gigabyte expanded CSV content into memory.

In [1]:
from __future__ import annotations

import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import PROCESSED_DIR, TABLE_DIR, FIGURE_DIR, CPSC_DIR, PMPM_PROXY_REVENUE

for path in [PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")

Project root: D:\Project 1\rcm-cms-mvp


## Chunked CPSC Aggregation

Why: CPSC enrollment files are large. The first safe aggregation level for the MVP is contract-plan-month and state-plan-month, not raw county rows. County-level CPSC is too large for a notebook loop and is unnecessary for the current dashboard because MA SCP already covers county opportunity.

In [2]:
def month_from_name(path: Path) -> str:
    match = re.search(r"(20\d{2}-\d{2})", path.name)
    if not match:
        raise ValueError(path.name)
    return match.group(1)


def find_csv(zf: zipfile.ZipFile, contains: str) -> str:
    matches = [n for n in zf.namelist() if contains.lower() in n.lower() and n.lower().endswith(".csv")]
    if not matches:
        raise ValueError(f"No CSV containing {contains}")
    return matches[0]


plan_rows = []
state_plan_rows = []
contract_rows = []
zip_paths = sorted(CPSC_DIR.glob("*.zip"))

for zip_path in zip_paths:
    month = month_from_name(zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        contract_csv = find_csv(zf, "Contract_Info")
        contract = pd.read_csv(zf.open(contract_csv), dtype=str, keep_default_na=False, encoding="cp1252")
        contract["report_month_label"] = month
        contract_rows.append(contract)

        enrollment_csv = find_csv(zf, "Enrollment_Info")
        chunks = pd.read_csv(
            zf.open(enrollment_csv),
            dtype=str,
            keep_default_na=False,
            encoding="cp1252",
            chunksize=250_000,
        )
        for chunk in chunks:
            chunk["Enrollment"] = pd.to_numeric(chunk["Enrollment"].str.replace(",", "", regex=False), errors="coerce")
            chunk["report_month_label"] = month
            plan_grouped = (
                chunk.groupby(["report_month_label", "Contract Number", "Plan ID"], as_index=False)
                .agg(observed_enrollment=("Enrollment", "sum"), source_rows=("Enrollment", "size"))
            )
            state_plan_grouped = (
                chunk.groupby(["report_month_label", "Contract Number", "Plan ID", "State"], as_index=False)
                .agg(observed_enrollment=("Enrollment", "sum"), source_rows=("Enrollment", "size"), counties=("FIPS State County Code", "nunique"))
            )
            plan_rows.append(plan_grouped)
            state_plan_rows.append(state_plan_grouped)

cpsc_plan_monthly = pd.concat(plan_rows, ignore_index=True)
cpsc_plan_monthly = (
    cpsc_plan_monthly.groupby(["report_month_label", "Contract Number", "Plan ID"], as_index=False)
    .agg(observed_enrollment=("observed_enrollment", "sum"), source_rows=("source_rows", "sum"))
)
cpsc_state_plan_monthly = pd.concat(state_plan_rows, ignore_index=True)
cpsc_state_plan_monthly = (
    cpsc_state_plan_monthly.groupby(["report_month_label", "Contract Number", "Plan ID", "State"], as_index=False)
    .agg(observed_enrollment=("observed_enrollment", "sum"), source_rows=("source_rows", "sum"), counties=("counties", "max"))
)
cpsc_contract = pd.concat(contract_rows, ignore_index=True)

cpsc_plan_monthly.to_parquet(PROCESSED_DIR / "cpsc_plan_monthly.parquet", index=False)
cpsc_state_plan_monthly.to_parquet(PROCESSED_DIR / "cpsc_state_plan_monthly.parquet", index=False)
cpsc_contract.to_parquet(PROCESSED_DIR / "cpsc_contract_info_monthly.parquet", index=False)
print(f"CPSC contract-plan monthly rows after aggregation: {len(cpsc_plan_monthly):,}")
print(f"CPSC state-plan monthly rows after aggregation: {len(cpsc_state_plan_monthly):,}")
print(f"CPSC contract rows: {len(cpsc_contract):,}")

CPSC contract-plan monthly rows after aggregation: 232,033
CPSC state-plan monthly rows after aggregation: 3,079,816
CPSC contract rows: 232,190


## Contract/Plan Growth Summary

Why: this identifies high-growth plans and contracts for RCM sales targeting.

In [3]:
plan_monthly = pd.read_parquet(PROCESSED_DIR / "cpsc_plan_monthly.parquet")
plan_monthly["report_month"] = pd.PeriodIndex(plan_monthly["report_month_label"], freq="M").to_timestamp()
plan_monthly = plan_monthly.sort_values(["Contract Number", "Plan ID", "report_month"])

plan_growth = (
    plan_monthly.groupby(["Contract Number", "Plan ID"], as_index=False)
    .agg(
        first_month=("report_month_label", "first"),
        last_month=("report_month_label", "last"),
        months_present=("report_month_label", "nunique"),
        first_enrollment=("observed_enrollment", "first"),
        last_enrollment=("observed_enrollment", "last"),
        source_rows=("source_rows", "sum"),
    )
)
plan_growth["absolute_growth"] = plan_growth["last_enrollment"] - plan_growth["first_enrollment"]
plan_growth["growth_pct"] = np.where(plan_growth["first_enrollment"] > 0, plan_growth["absolute_growth"] / plan_growth["first_enrollment"], np.nan)

latest_contract = cpsc_contract.sort_values("report_month_label").groupby(["Contract ID", "Plan ID"], as_index=False).tail(1)
plan_growth = plan_growth.merge(
    latest_contract[["Contract ID", "Plan ID", "Organization Name", "Plan Name", "Plan Type", "Parent Organization"]],
    left_on=["Contract Number", "Plan ID"],
    right_on=["Contract ID", "Plan ID"],
    how="left",
).sort_values(["absolute_growth", "last_enrollment"], ascending=False)

plan_monthly.to_parquet(PROCESSED_DIR / "cpsc_plan_monthly.parquet", index=False)
plan_growth.to_csv(TABLE_DIR / "cpsc_plan_growth.csv", index=False)
display(plan_growth.head(25))

,Contract Number,Plan ID,first_month,last_month,months_present,first_enrollment,last_enrollment,source_rows,absolute_growth,growth_pct,Contract ID,Organization Name,Plan Name,Plan Type,Parent Organization
10576,S8841,821,2025-01,2026-05,17,0.0,433271.0,53935,433271.0,NaN,S8841,"OPTUM INSURANCE OF OHIO, INC.",URMBT 2026 - CY (PDP),Medicare Prescription Drug Plan,"UnitedHealth Group, Inc."
10303,S5884,834,2024-01,2026-05,29,40858.0,372522.0,90249,331664.0,8.117480,S5884,Humana Insurance Co. & Humana Insurance Co. of NY,Humana Medicare Employer (PDP),Medicare Prescription Drug Plan,Humana Inc.
9946,S5617,158,2024-01,2026-05,29,236730.0,553383.0,30475,316653.0,1.337612,S5617,MEDCO CONTAINMENT LIFE AND MEDCO CONTAINMENT NY,HealthSpring Assurance Rx (PDP),Medicare Prescription Drug Plan,Health Care Service Corporation
10192,S5820,834,2024-01,2026-05,29,455.0,269251.0,90247,268796.0,590.760440,S5820,UNITEDHEALTHCARE INS. CO. & UHC INS. CO. OF NY,UnitedHealthcare MedicareRx for Groups (PDP),Medicare Prescription Drug Plan,"UnitedHealth Group, Inc."
9689,S4802,146,2024-01,2026-05,29,239140.0,495391.0,21260,256251.0,1.071552,S4802,"WELLCARE PRESCRIPTION INSURANCE, INC.",Wellcare Value Script (PDP),Medicare Prescription Drug Plan,Centene Corporation
9698,S4802,155,2024-01,2026-05,29,212451.0,462966.0,16869,250515.0,1.179166,S4802,"WELLCARE PRESCRIPTION INSURANCE, INC.",Wellcare Value Script (PDP),Medicare Prescription Drug Plan,Centene Corporation
5658,H5216,806,2024-01,2026-05,29,33202.0,280296.0,90601,247094.0,7.442142,H5216,HUMANA INSURANCE COMPANY,Humana Medicare Employer (PPO),Local PPO,Humana Inc.
2202,H2001,817,2024-01,2026-05,29,503911.0,739143.0,94879,235232.0,0.466813,H2001,"SIERRA HEALTH AND LIFE INSURANCE COMPANY, INC.",UnitedHealthcare Group Medicare Advantage (PPO),Local PPO,"UnitedHealth Group, Inc."
9706,S4802,163,2024-01,2026-05,29,286945.0,462798.0,10596,175853.0,0.612846,S4802,"WELLCARE PRESCRIPTION INSURANCE, INC.",Wellcare Value Script (PDP),Medicare Prescription Drug Plan,Centene Corporation
9912,S5601,813,2024-01,2026-05,29,244074.0,402257.0,93092,158183.0,0.648094,S5601,SILVERSCRIPT INSURANCE COMPANY,SilverScript Group SF (PDP),Medicare Prescription Drug Plan,CVS Health Corporation


## CPSC Visual Results

Elements: bars show absolute growth; color shows plan type where available. These are optional deeper sales leads beyond the national forecast.

In [4]:
top_plans = plan_growth.head(20).copy()
top_plans["plan_label"] = top_plans["Contract Number"] + "-" + top_plans["Plan ID"]
fig = px.bar(top_plans, x="plan_label", y="absolute_growth", color="Plan Type", hover_name="Plan Name", title="Top CPSC Plans by Absolute Enrollment Growth")
fig.show()

plan_growth_scatter = plan_growth.replace([np.inf, -np.inf], np.nan).dropna(subset=["growth_pct"]).head(500).copy()
plan_growth_scatter["abs_growth_for_marker"] = plan_growth_scatter["absolute_growth"].abs().clip(lower=1)
fig = px.scatter(
    plan_growth_scatter,
    x="last_enrollment",
    y="growth_pct",
    size="abs_growth_for_marker",
    hover_name="Plan Name",
    color="Plan Type",
    title="CPSC Plan Scale vs Growth",
)
fig.show()

state_plan = pd.read_parquet(PROCESSED_DIR / "cpsc_state_plan_monthly.parquet")
latest_month = state_plan["report_month_label"].max()
latest_state_plan = (
    state_plan[state_plan["report_month_label"].eq(latest_month)]
    .groupby("State", as_index=False)
    .agg(observed_enrollment=("observed_enrollment", "sum"), plans=("Plan ID", "nunique"))
    .sort_values("observed_enrollment", ascending=False)
    .head(25)
)
latest_state_plan.to_csv(TABLE_DIR / "cpsc_latest_state_summary.csv", index=False)
fig = px.bar(latest_state_plan, x="State", y="observed_enrollment", title=f"CPSC Latest Enrollment by State ({latest_month})")
fig.show()